# 🖋️ Arabic Handwriting OCR — v5 الموحدة (جاهزة لـ Colab)
> إصدار واحد يجمع كل التحسينات: معالجة صفحة بصفحة | TrOCR + EasyOCR | LoRA Fine‑tuning | Active Learning | Study Guide | Gradio UI

In [1]:
# ============================================================
# الخلية 1: تثبيت المكتبات على Colab (تشغّل مرة واحدة)
# ============================================================
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q \
    pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm fastapi uvicorn pyngrok

# لـ Manjaro/Arch محلياً (أزل التعليق):
# !sudo pacman -S python-pip poppler tesseract tesseract-data-ara --noconfirm
# !pip install --user pdf2image easyocr ...

Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-ara.
Preparing to unpack .../tesseract-ocr-ara_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# الخلية 2: الاستيرادات الأساسية
# ============================================================
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
import socket, fcntl
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import gradio as gr
import easyocr
from PIL import Image
from pdf2image import convert_from_path, pdfinfo_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from tqdm import tqdm

# مدقق عربي اختياري
try:
    from ar_corrector.corrector import Corrector
    _AR_OK = True
except ImportError:
    _AR_OK = False

# تجهيز بيئة Colab
try:
    from google.colab import drive, userdata
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    _HF_TOKEN = userdata.get('HF_TOKEN')
    if _HF_TOKEN:
        os.environ['HF_TOKEN'] = _HF_TOKEN
    IN_COLAB = True
    print('✅ Colab + Drive + HF_TOKEN')
except ImportError:
    _HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ بيئة محلية')

DetectorFactory.seed = 0
print('✅ الاستيرادات اكتملت')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ============================================================
# الخلية 3: الإعدادات المركزية (Config)
# ============================================================
@dataclass
class Config:
    project_root: str  = '/content/drive/MyDrive/Handwritten_OCR_Ultimate'
    pdf_path: str      = '/content/drive/MyDrive/input.pdf'
    db_name: str       = 'handwriting_data.db'
    trocr_model_name: str = 'David-Magdy/TR_OCR_LARGE'
    hf_token: str         = ''
    hf_dataset_repo: str  = ''
    ocr_languages: list   = field(default_factory=lambda: ['en', 'ar'])
    memory_mode: str   = 'auto'
    use_gpu: bool      = True
    use_fp16: bool     = True
    trocr_batch_size: int      = 4
    num_beams: int             = 4
    max_text_length: int       = 64
    easy_conf_threshold: float = 0.80
    trocr_default_conf: float  = 0.70
    low_conf_threshold: float  = 0.50
    dpi: int            = 300
    clahe_clip: float   = 2.0
    clahe_tile: tuple   = (8, 8)
    denoise_h: int      = 20
    enable_deskew: bool = True
    min_word_w: int     = 15
    min_word_h: int     = 10
    dilation_kernel: tuple = (25, 5)
    correction_min_votes: int = 1
    pages_start: int = 1
    pages_end: int   = 5
    finetune_min_samples: int  = 50
    finetune_epochs: int       = 5
    finetune_batch_size: int   = 4
    finetune_lr: float         = 1e-5
    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.1
    lora_targets: list         = field(default_factory=lambda: ['query', 'value'])
    al_min_corrections: int    = 50
    al_reprocess_limit: int    = 100
    sync_lock_timeout: int     = 30
    export_val_ratio: float    = 0.1
    auto_export: bool          = True
    gradio_share: bool = True
    gradio_port: int   = 7860
    log_level: str = 'INFO'

    @property
    def root(self):         return Path(self.project_root)
    @property
    def db_path(self):      return self.root / 'database' / self.db_name
    @property
    def logs_dir(self):     return self.root / 'logs'
    @property
    def exports_dir(self):  return self.root / 'exports'
    @property
    def cache_dir(self):    return self.root / 'models_cache'
    @property
    def artifacts_dir(self):return self.root / 'artifacts'
    @property
    def backups_dir(self):  return self.root / 'backups'
    @property
    def tb_dir(self):       return self.root / 'runs'
    @property
    def feedback_csv(self): return self.logs_dir / 'user_corrections_feedback.csv'
    @property
    def stats_json(self):   return self.logs_dir / 'processing_stats.json'
    @property
    def correction_dict_path(self): return self.artifacts_dir / 'correction_dict.json'
    @property
    def checkpoint_file(self): return self.artifacts_dir / 'ocr_checkpoint.json'
    @property
    def metrics_log(self):  return self.logs_dir / 'metrics_history.csv'
    @property
    def runs_csv(self):     return self.logs_dir / 'runs_history.csv'
    @property
    def lora_path(self):    return self.cache_dir / 'trocr_lora_finetuned'
    @property
    def lock_file(self):    return self.artifacts_dir / 'ocr.lock'
    @property
    def sync_status(self):  return self.artifacts_dir / 'sync_status.json'
    @property
    def study_guide_dir(self): return self.exports_dir / 'study_guide'

    def resolve_memory_mode(self):
        if self.memory_mode != 'auto':
            return self.memory_mode
        if torch.cuda.is_available():
            vram = torch.cuda.get_device_properties(0).total_memory // 1e9
            return 'high' if vram >= 8 else 'low'
        import psutil
        ram_gb = psutil.virtual_memory().total / 1e9
        return 'high' if ram_gb >= 24 else 'low'

    def setup(self):
        for d in [self.root/'database', self.logs_dir, self.exports_dir,
                  self.cache_dir, self.artifacts_dir, self.backups_dir,
                  self.tb_dir, self.study_guide_dir]:
            d.mkdir(parents=True, exist_ok=True)
        if self.hf_token:
            os.environ['HF_TOKEN'] = self.hf_token
        os.environ['TRANSFORMERS_CACHE'] = str(self.cache_dir)
        os.environ['TORCH_HOME']         = str(self.cache_dir)
        for csv_path, cols in [
            (self.feedback_csv, ['timestamp','image_id','original_text','corrected_text','status']),
            (self.runs_csv, ['run_id','timestamp','pages','words','avg_conf','duration_sec','status']),
        ]:
            if not csv_path.exists():
                pd.DataFrame(columns=cols).to_csv(csv_path, index=False, encoding='utf-8-sig')
        mode = self.resolve_memory_mode()
        if self.trocr_batch_size == 4:
            self.trocr_batch_size = 16 if mode == 'high' else 1
        print(f'💾 وضع الذاكرة: {mode} | batch_size={self.trocr_batch_size}')

    def setup_easyocr_symlink(self):
        local = Path.home() / '.EasyOCR'
        drive_p = self.root / '.EasyOCR'
        if not drive_p.exists():
            drive_p.mkdir(parents=True, exist_ok=True)
        if local.exists() and not local.is_symlink():
            shutil.move(str(local), str(drive_p))
        if not local.exists():
            try:
                os.symlink(drive_p, local)
            except Exception:
                pass

    @classmethod
    def for_colab(cls, pdf_name='input.pdf', hf_token='', hf_repo='',
                  memory_mode='auto'):
        return cls(
            project_root='/content/drive/MyDrive/Handwritten_OCR_Ultimate',
            pdf_path=f'/content/drive/MyDrive/{pdf_name}',
            hf_token=hf_token or _HF_TOKEN or '',
            hf_dataset_repo=hf_repo,
            memory_mode=memory_mode,
        )

    @classmethod
    def for_local(cls, base_dir='~/handwriting_ocr', pdf_path='./input.pdf',
                  memory_mode='low'):
        return cls(
            project_root=str(Path(base_dir).expanduser()),
            pdf_path=str(Path(pdf_path).expanduser()),
            memory_mode=memory_mode,
            use_fp16=False,
            dpi=200,
            pages_end=3,
        )

CFG = Config.for_colab(pdf_name='python notes.pdf')
# للمحلي: CFG = Config.for_local(base_dir='~/handwriting_ocr', pdf_path='./input.pdf')
CFG.setup()
CFG.setup_easyocr_symlink()
print('✅ Config جاهز:', CFG.root)

In [ ]:
# ============================================================
# الخلية 4: نظام اللوغات
# ============================================================
_LOG_FILE = CFG.logs_dir / f'ocr_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
_EVENTS   = CFG.logs_dir / 'ocr_events.jsonl'

LOGGER = logging.getLogger('HandwrittenOCR')
LOGGER.setLevel(getattr(logging, CFG.log_level))
LOGGER.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for _h in [
    RotatingFileHandler(_LOG_FILE, maxBytes=2_000_000, backupCount=5, encoding='utf-8'),
    logging.StreamHandler()
]:
    _h.setFormatter(_fmt)
    LOGGER.addHandler(_h)

def log_event(event_type: str, payload: dict = None, level: str = 'info'):
    payload = payload or {}
    entry = {'ts': datetime.now().isoformat(), 'event': event_type, 'data': payload}
    with open(_EVENTS, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    getattr(LOGGER, level, LOGGER.info)(
        f'{event_type} | {json.dumps(payload, ensure_ascii=False)}')

print('✅ اللوغات جاهزة')

In [ ]:
# ============================================================
# الخلية 5: قفل الملفات (FileLock)
# ============================================================
class FileLock:
    def __init__(self, lock_path, timeout=30):
        self.lock_path = Path(lock_path)
        self.timeout   = timeout
        self._file     = None

    def acquire(self):
        self.lock_path.parent.mkdir(parents=True, exist_ok=True)
        self._file = open(self.lock_path, 'w')
        start = time.time()
        while True:
            try:
                if platform.system() == 'Windows':
                    import msvcrt
                    msvcrt.locking(self._file.fileno(), msvcrt.LK_NBLCK, 1)
                else:
                    fcntl.flock(self._file, fcntl.LOCK_EX | fcntl.LOCK_NB)
                info = {'pid': os.getpid(), 'host': socket.gethostname(),
                        'ts': datetime.now().isoformat()}
                self._file.seek(0); self._file.truncate()
                self._file.write(json.dumps(info)); self._file.flush()
                return True
            except (BlockingIOError, OSError):
                if time.time() - start > self.timeout:
                    self._file.close(); self._file = None
                    raise TimeoutError('قفل الملف مشغول — جهاز آخر يعمل.')
                time.sleep(0.5)

    def release(self):
        if self._file is None: return
        try:
            if platform.system() == 'Windows':
                import msvcrt; msvcrt.locking(self._file.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(self._file, fcntl.LOCK_UN)
        except Exception: pass
        finally:
            try: self._file.close()
            except Exception: pass
            self._file = None

    def __enter__(self): self.acquire(); return self
    def __exit__(self, *_): self.release(); return False

print('✅ FileLock جاهز')

In [ ]:
# ============================================================
# الخلية 6: قاعدة البيانات DB
# ============================================================
class HandwritingDB:
    SCHEMA = '''
        CREATE TABLE IF NOT EXISTS handwriting_data (
            image_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            image_data     BLOB    NOT NULL,
            predicted_text TEXT    DEFAULT '',
            raw_text       TEXT    DEFAULT '',
            status         TEXT    DEFAULT 'unverified',
            confidence     REAL    DEFAULT 0.0,
            model_source   TEXT    DEFAULT 'none',
            x INTEGER DEFAULT 0, y INTEGER DEFAULT 0,
            w INTEGER DEFAULT 0, h INTEGER DEFAULT 0,
            page_num       INTEGER DEFAULT 0,
            run_id         TEXT    DEFAULT '',
            created_at     TEXT    DEFAULT '',
            updated_at     TEXT    DEFAULT ''
        );
        CREATE TABLE IF NOT EXISTS processing_runs (
            run_id           TEXT PRIMARY KEY,
            started_at       TEXT,
            ended_at         TEXT,
            input_path       TEXT,
            pages_processed  INTEGER DEFAULT 0,
            words_processed  INTEGER DEFAULT 0,
            avg_confidence   REAL    DEFAULT 0.0,
            status           TEXT    DEFAULT 'running'
        );
        CREATE TABLE IF NOT EXISTS review_events (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp      TEXT,
            image_id       INTEGER,
            original_text  TEXT,
            corrected_text TEXT,
            action         TEXT,
            reviewer       TEXT DEFAULT 'user'
        );
        CREATE INDEX IF NOT EXISTS idx_status ON handwriting_data(status);
        CREATE INDEX IF NOT EXISTS idx_page   ON handwriting_data(page_num);
        CREATE INDEX IF NOT EXISTS idx_conf   ON handwriting_data(confidence);
        CREATE INDEX IF NOT EXISTS idx_run    ON handwriting_data(run_id);
    '''

    def __init__(self, db_path: Path):
        self.db_path = db_path
        db_path.parent.mkdir(parents=True, exist_ok=True)
        with self._conn() as c:
            c.executescript(self.SCHEMA)
            self._migrate(c)
        LOGGER.info(f'DB جاهزة: {db_path}')

    def _conn(self):
        conn = sqlite3.connect(self.db_path, check_same_thread=False)
        conn.execute('PRAGMA journal_mode=WAL')
        conn.execute('PRAGMA synchronous=NORMAL')
        return conn

    def _migrate(self, conn):
        cur = conn.execute("PRAGMA table_info(handwriting_data)")
        existing = {r[1] for r in cur.fetchall()}
        new_cols = {
            'raw_text':   "TEXT DEFAULT ''",
            'run_id':     "TEXT DEFAULT ''",
            'created_at': "TEXT DEFAULT ''",
            'updated_at': "TEXT DEFAULT ''",
        }
        for col, typedef in new_cols.items():
            if col not in existing:
                conn.execute(f"ALTER TABLE handwriting_data ADD COLUMN {col} {typedef}")
        conn.execute("UPDATE handwriting_data SET status='verified'   WHERE status='yes'")
        conn.execute("UPDATE handwriting_data SET status='unverified' WHERE status='no'")
        conn.commit()

    def insert_word(self, image_data, predicted_text, raw_text='',
                    status='unverified', confidence=0.0, model_source='none',
                    x=0, y=0, w=0, h=0, page_num=0, run_id='') -> int:
        ts = datetime.now().isoformat()
        with self._conn() as c:
            cur = c.execute(
                '''INSERT INTO handwriting_data
                   (image_data,predicted_text,raw_text,status,confidence,
                    model_source,x,y,w,h,page_num,run_id,created_at,updated_at)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                (image_data, predicted_text, raw_text, status, confidence,
                 model_source, x, y, w, h, page_num, run_id, ts, ts))
            c.commit()
            return cur.lastrowid

    def update_word(self, image_id, predicted_text=None, status=None):
        sets, vals = ['updated_at=?'], [datetime.now().isoformat()]
        if predicted_text is not None:
            sets.insert(0, 'predicted_text=?'); vals.insert(0, predicted_text)
        if status is not None:
            sets.insert(0, 'status=?'); vals.insert(0, status)
        vals.append(image_id)
        with self._conn() as c:
            c.execute(f"UPDATE handwriting_data SET {','.join(sets)} WHERE image_id=?", vals)
            c.commit()

    def delete_word(self, image_id: int):
        with self._conn() as c:
            c.execute('DELETE FROM handwriting_data WHERE image_id=?', (image_id,))
            c.commit()

    def delete_pages(self, start, end) -> int:
        with self._conn() as c:
            cur = c.execute(
                'DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?', (start, end))
            c.commit()
            return cur.rowcount

    def _rows(self, sql, params=()):
        with self._conn() as c:
            c.row_factory = sqlite3.Row
            return [dict(r) for r in c.execute(sql, params).fetchall()]

    def get_all(self):
        return self._rows('SELECT * FROM handwriting_data ORDER BY page_num,y,x')

    def get_unverified(self):
        return self._rows(
            "SELECT * FROM handwriting_data WHERE status='unverified' "
            "ORDER BY confidence ASC")

    def get_verified(self):
        return self._rows(
            "SELECT * FROM handwriting_data "
            "WHERE status IN ('verified','sentence_corrected') ORDER BY image_id")

    def get_low_confidence(self, threshold=0.5, limit=100):
        return self._rows(
            "SELECT * FROM handwriting_data WHERE confidence<? "
            "ORDER BY confidence ASC LIMIT ?", (threshold, limit))

    def count_by_status(self) -> dict:
        rows = self._rows(
            'SELECT status, COUNT(*) as cnt FROM handwriting_data GROUP BY status')
        return {r['status']: r['cnt'] for r in rows}

    def insert_run(self, run_id, input_path):
        with self._conn() as c:
            c.execute(
                'INSERT OR REPLACE INTO processing_runs '
                '(run_id,started_at,input_path,status) VALUES (?,?,?,?)',
                (run_id, datetime.now().isoformat(), str(input_path), 'running'))
            c.commit()

    def finish_run(self, run_id, pages, words, avg_conf):
        with self._conn() as c:
            c.execute(
                'UPDATE processing_runs SET ended_at=?,pages_processed=?,'
                'words_processed=?,avg_confidence=?,status=? WHERE run_id=?',
                (datetime.now().isoformat(), pages, words, avg_conf, 'completed', run_id))
            c.commit()

    def log_review(self, image_id, original, corrected, action):
        with self._conn() as c:
            c.execute(
                'INSERT INTO review_events '
                '(timestamp,image_id,original_text,corrected_text,action) VALUES (?,?,?,?,?)',
                (datetime.now().isoformat(), image_id, original, corrected, action))
            c.commit()

    def get_last_review_event(self):
        with self._conn() as c:
            c.row_factory = sqlite3.Row
            row = c.execute(
                "SELECT * FROM review_events WHERE action IN ('verified') AND image_id IS NOT NULL ORDER BY timestamp DESC LIMIT 1"
            ).fetchone()
            return dict(row) if row else None

    def delete_review_event(self, event_id: int):
        with self._conn() as c:
            c.execute('DELETE FROM review_events WHERE id=?', (event_id,))
            c.commit()

DB = HandwritingDB(CFG.db_path)
print('✅ قاعدة البيانات جاهزة')

In [ ]:
# ============================================================
# الخلية 7: الترحيل الذكي (SmartMigrator)
# ============================================================
class SmartMigrator:
    OLD_PROJECT_NAMES = [
        'Handwriting_Dataset', 'Handwritten_OCR_Integrated',
        'Handwritten_OCR_Pro', 'Handwritten_OCR', 'Arabic_OCR_Pro',
        'Arabic_OCR', 'HandwrittenOCR', 'Handwritten_OCR_Ultimate',
    ]
    IMPORT_STATUSES = ('verified', 'sentence_corrected', 'yes')

    def __init__(self, config):
        self.cfg = config
        self.report = {
            'scanned_projects': [],
            'imported_words': 0,
            'imported_feedback': 0,
            'dict_entries': 0,
            'deleted_folders': [],
            'skipped_folders': [],
            'errors': [],
        }

    def scan(self, base_path=None):
        base = Path(base_path or self.cfg.root.parent)
        found = []
        for name in self.OLD_PROJECT_NAMES:
            folder = base / name
            if not folder.exists() or folder == self.cfg.root:
                continue
            info = {'name': name, 'path': str(folder),
                    'size_mb': self._folder_size_mb(folder),
                    'db_files': [], 'feedback_files': [], 'correction_dicts': []}
            for db_f in folder.rglob('*.db'):
                wc = self._count_verified_words(str(db_f))
                info['db_files'].append({'path': str(db_f), 'verified_words': wc})
            for pat in ['*feedback*.csv', '*correction*.csv']:
                for f in folder.rglob(pat):
                    try:
                        n = len(pd.read_csv(f, encoding='utf-8-sig'))
                        info['feedback_files'].append({'path': str(f), 'rows': n})
                    except Exception: pass
            for jf in folder.rglob('correction_dict.json'):
                try:
                    d = json.loads(Path(jf).read_text('utf-8'))
                    info['correction_dicts'].append({'path': str(jf), 'entries': len(d)})
                except Exception: pass
            total_words = sum(d['verified_words'] for d in info['db_files'])
            total_feedback = sum(f['rows'] for f in info['feedback_files'])
            if total_words > 0 or total_feedback > 0:
                info['total_verified_words'] = total_words
                info['total_feedback_rows'] = total_feedback
                found.append(info)
        self.report['scanned_projects'] = found
        return found

    def migrate(self, base_path=None, delete_after=False, dry_run=False):
        if not self.report['scanned_projects']:
            self.scan(base_path)
        if not self.report['scanned_projects']:
            print('⚠️  لم يتم العثور على مشاريع قديمة.')
            return self.report
        if dry_run:
            print('[dry_run] لن يتم أي تغيير.')
            return self.report
        for proj in self.report['scanned_projects']:
            for db_info in proj['db_files']:
                if db_info['verified_words'] > 0:
                    try:
                        n = self._import_db(db_info['path'], proj['name'])
                        self.report['imported_words'] += n
                        print(f"  ✅ {proj['name']}: {n} كلمة")
                    except Exception as e:
                        self.report['errors'].append(str(e))
        self.report['imported_feedback'] = self._merge_feedback_files()
        self.report['dict_entries'] = self._merge_correction_dicts()
        global correction_dict
        correction_dict = build_correction_dict()
        if delete_after:
            self._safe_delete()
        self._print_summary()
        return self.report

    def _count_verified_words(self, db_path):
        try:
            with sqlite3.connect(db_path) as c:
                cur = c.execute("SELECT COUNT(*) FROM handwriting_data WHERE status IN ('verified','sentence_corrected','yes')")
                return cur.fetchone()[0]
        except Exception: return 0

    def _folder_size_mb(self, folder):
        total = sum(f.stat().st_size for f in folder.rglob('*') if f.is_file())
        return round(total / (1024 * 1024), 1)

    def _import_db(self, src_path, label):
        src = sqlite3.connect(src_path)
        tgt = sqlite3.connect(str(self.cfg.db_path))
        try:
            cur = src.execute("PRAGMA table_info(handwriting_data)")
            src_cols = {r[1] for r in cur.fetchall()}
            wanted = ['image_data','predicted_text','raw_text','status','confidence',
                      'model_source','x','y','w','h','page_num']
            avail = [c for c in wanted if c in src_cols]
            rows = src.execute(f"SELECT {','.join(avail)} FROM handwriting_data WHERE status IN ('verified','sentence_corrected','yes')").fetchall()
            now = datetime.now().isoformat()
            tgt.execute('PRAGMA journal_mode=WAL')
            cur_tgt = tgt.cursor()
            cnt = 0
            for row in rows:
                d = dict(zip(avail, row))
                d['status'] = 'verified' if d.get('status') == 'yes' else d.get('status','verified')
                predicted = str(d.get('predicted_text','') or '').strip()
                dup = cur_tgt.execute("SELECT 1 FROM handwriting_data WHERE predicted_text=? AND page_num=? AND x=? AND y=? LIMIT 1",
                                      (predicted, d.get('page_num',0) or 0, d.get('x',0) or 0, d.get('y',0) or 0)).fetchone()
                if dup: continue
                cur_tgt.execute('''INSERT INTO handwriting_data
                    (image_data,predicted_text,raw_text,status,confidence,
                     model_source,x,y,w,h,page_num,run_id,created_at,updated_at)
                    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                    (d.get('image_data',b''), predicted,
                     str(d.get('raw_text',predicted)), d['status'],
                     float(d.get('confidence',0.9)), d.get('model_source','migrated'),
                     d.get('x',0) or 0, d.get('y',0) or 0,
                     d.get('w',0) or 0, d.get('h',0) or 0,
                     d.get('page_num',0) or 0,
                     f'migrated_{label}_{datetime.now().strftime("%Y%m%d")}',
                     now, now))
                cnt += 1
            tgt.commit()
            return cnt
        finally:
            src.close(); tgt.close()

    def _merge_feedback_files(self):
        all_dfs = []
        for proj in self.report['scanned_projects']:
            for fb in proj['feedback_files']:
                try:
                    df = pd.read_csv(fb['path'], encoding='utf-8-sig')
                    col_map = {}
                    for c in df.columns:
                        if 'original' in c.lower(): col_map[c] = 'original_text'
                        elif 'correct' in c.lower(): col_map[c] = 'corrected_text'
                    if col_map: df = df.rename(columns=col_map)
                    if {'original_text','corrected_text'}.issubset(df.columns):
                        all_dfs.append(df)
                except Exception: pass
        if not all_dfs: return 0
        merged = pd.concat(all_dfs, ignore_index=True)
        merged = merged.dropna(subset=['original_text','corrected_text'])
        merged = merged[merged['original_text'].astype(str).str.strip() != '']
        merged = merged[merged['original_text'].astype(str) != merged['corrected_text'].astype(str)]
        merged = merged.drop_duplicates(subset=['original_text','corrected_text'], keep='last')
        if self.cfg.feedback_csv.exists():
            existing = pd.read_csv(self.cfg.feedback_csv, encoding='utf-8-sig')
            merged = pd.concat([existing, merged], ignore_index=True)
            merged = merged.drop_duplicates(subset=['original_text','corrected_text'], keep='last')
        merged.to_csv(self.cfg.feedback_csv, index=False, encoding='utf-8-sig')
        print(f'  📋 feedback: {len(merged)} إدخال')
        return len(merged)

    def _merge_correction_dicts(self):
        buckets = defaultdict(Counter)
        for proj in self.report['scanned_projects']:
            for d_info in proj['correction_dicts']:
                try:
                    d = json.loads(Path(d_info['path']).read_text('utf-8'))
                    for o,c in d.items(): buckets[o][c] += 1
                except Exception: pass
        if self.cfg.feedback_csv.exists():
            df = pd.read_csv(self.cfg.feedback_csv, encoding='utf-8-sig')
            for _, row in df.iterrows():
                o = str(row.get('original_text','')).strip()
                c = str(row.get('corrected_text','')).strip()
                if o and c and o!=c: buckets[o][c] += 2
        merged_dict = {o: cnt.most_common(1)[0][0] for o,cnt in buckets.items()
                       if cnt.most_common(1)[0][1]>=1}
        if self.cfg.correction_dict_path.exists():
            cur = json.loads(self.cfg.correction_dict_path.read_text('utf-8'))
            merged_dict = {**merged_dict, **cur}
        self.cfg.correction_dict_path.write_text(json.dumps(merged_dict, ensure_ascii=False, indent=2))
        return len(merged_dict)

    def _safe_delete(self):
        if self.report['imported_words']==0 and self.report['imported_feedback']==0:
            print('⚠️  لا شيء للحذف.'); return
        for proj in self.report['scanned_projects']:
            folder = Path(proj['path'])
            if folder.resolve() == self.cfg.root.resolve():
                self.report['skipped_folders'].append(proj['path'])
                continue
            # نسخة احتياطية
            backup_name = self.cfg.backups_dir / f"pre_delete_{proj['name']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
            try:
                import zipfile
                with zipfile.ZipFile(backup_name, 'w', zipfile.ZIP_DEFLATED) as zf:
                    for f in folder.rglob('*.db'): zf.write(f, f.relative_to(folder))
                    for f in folder.rglob('*.csv'): zf.write(f, f.relative_to(folder))
                    for f in folder.rglob('*.json'): zf.write(f, f.relative_to(folder))
                shutil.rmtree(folder)
                self.report['deleted_folders'].append(proj['path'])
                print(f'  ✅ حُذف: {proj["name"]}')
            except Exception as e:
                self.report['errors'].append(f"حذف {proj['name']}: {e}")

    def _print_summary(self):
        r = self.report
        print(f"📥 كلمات مُستوردة: {r['imported_words']}")
        print(f"📋 feedback: {r['imported_feedback']}")
        print(f"📖 قاموس: {r['dict_entries']}")
        print(f"🗑️ مجلدات حُذفت: {len(r['deleted_folders'])}")

SMART_MIGRATOR = SmartMigrator(CFG)
print('✅ SmartMigrator جاهز')
print('استخدم: SMART_MIGRATOR.migrate(delete_after=True) لترحيل قديم')

In [ ]:
# ============================================================
# الخلية 8: Preprocessing
# ============================================================
def deskew(gray):
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50: return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3: return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def preprocess_image(img_bgr, adaptive=False):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim==3 else img_bgr.copy()
    if CFG.enable_deskew: gray = deskew(gray)
    gray = cv2.createCLAHE(clipLimit=CFG.clahe_clip, tileGridSize=CFG.clahe_tile).apply(gray)
    gray = cv2.fastNlMeansDenoising(gray, h=CFG.denoise_h)
    if adaptive:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary, gray

def _iou(b1, b2):
    x1,y1,w1,h1 = b1; x2,y2,w2,h2 = b2
    xi1,yi1 = max(x1,x2), max(y1,y2)
    xi2,yi2 = min(x1+w1,x2+w2), min(y1+h1,y2+h2)
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    union = w1*h1+w2*h2-inter
    return inter/union if union>0 else 0

def smart_segmentation(img_bgr, binary, detections=None):
    if detections:
        boxes = []
        for det in detections:
            pts = np.array(det[0], dtype=np.int32)
            x,y,w,h = cv2.boundingRect(pts)
            if w>CFG.min_word_w and h>CFG.min_word_h:
                boxes.append((x,y,w,h))
        if boxes: return boxes
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, CFG.dilation_kernel)
    dilated = cv2.dilate(binary, kernel, iterations=1)
    contours,_ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [(x,y,w,h) for c in contours for x,y,w,h in [cv2.boundingRect(c)]
             if w>CFG.min_word_w and h>CFG.min_word_h]
    return sorted(boxes, key=lambda b: (b[1],b[0]))

def match_boxes_detections(boxes, detections):
    if not detections: return [(b, None) for b in boxes]
    det_boxes = []
    for det in detections:
        pts = np.array(det[0], dtype=np.int32)
        det_boxes.append((cv2.boundingRect(pts), det))
    result, used = [], set()
    for box in boxes:
        best_det, best_iou = None, 0
        for i, (db, det) in enumerate(det_boxes):
            if i in used: continue
            iou = _iou(box, db)
            if iou > best_iou and iou > 0.3:
                best_iou, best_det = iou, det
                used.add(i)
        result.append((box, best_det))
    return result

def crop_safe(img, x, y, w, h):
    H,W = img.shape[:2]
    return img[max(0,y):min(H,y+h), max(0,x):min(W,x+w)]

print('✅ Preprocessing جاهز')

In [ ]:
# ============================================================
# الخلية 9: OCR Engine
# ============================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() and CFG.use_gpu else 'cpu')
LOGGER.info(f'Device: {DEVICE}')
print(f'⏳ تحميل النماذج على {DEVICE}...')
_t0 = time.time()

if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, os.cpu_count() or 2))

_hf_kw = {'cache_dir': str(CFG.cache_dir)}
if CFG.hf_token:
    _hf_kw['token'] = CFG.hf_token

# تفريغ الكاش القديم لتجنب أخطاء الإدخال/الإخراج
if CFG.cache_dir.exists():
    shutil.rmtree(CFG.cache_dir)
    CFG.cache_dir.mkdir(parents=True, exist_ok=True)

original_cache = os.environ.get('TRANSFORMERS_CACHE')
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_transformers_cache'
Path('/tmp/hf_transformers_cache').mkdir(parents=True, exist_ok=True)

try:
    trocr_processor = TrOCRProcessor.from_pretrained(CFG.trocr_model_name, **_hf_kw)
    trocr_model     = VisionEncoderDecoderModel.from_pretrained(CFG.trocr_model_name, **_hf_kw)
finally:
    if original_cache:
        os.environ['TRANSFORMERS_CACHE'] = original_cache
    else:
        del os.environ['TRANSFORMERS_CACHE']

if DEVICE.type == 'cuda' and CFG.use_fp16:
    trocr_model = trocr_model.half()
    print('✅ FP16 مفعّل')
trocr_model = trocr_model.to(DEVICE)

if CFG.lora_path.exists():
    try:
        from peft import PeftModel
        trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_path)).to(DEVICE)
        print('✅ LoRA محمّلة')
    except Exception as e:
        print(f'⚠️ LoRA: {e}')

try:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=(DEVICE.type=='cuda'))
except Exception:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=False)

_ar_corrector  = Corrector() if _AR_OK else None
_en_spellcheck = SpellChecker(language='en')
print(f'✅ النماذج جاهزة في {time.time()-_t0:.1f}s')

def normalize_text(x):
    return '' if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x).strip()

def detect_lang(text):
    try: return detect(str(text)) if text and text.strip() else 'unknown'
    except Exception: return 'unknown'

def spell_correct(text):
    text = normalize_text(text)
    if not text: return ''
    try:
        lang = detect_lang(text)
        if lang == 'ar' and _ar_corrector:
            return _ar_corrector.contextual_correct(text)
        return ' '.join(_en_spellcheck.correction(w) or w for w in text.split())
    except Exception: return text

def trocr_batch_predict(crops):
    if not crops: return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops]
    try:
        enc = trocr_processor(images=pil_imgs, return_tensors='pt', padding=True)
        pv  = enc.pixel_values
        if CFG.use_fp16 and DEVICE.type == 'cuda':
            pv = pv.half()
        pv = pv.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(pv, max_length=CFG.max_text_length,
                                       num_beams=CFG.num_beams, early_stopping=True,
                                       length_penalty=1.0)
        texts = trocr_processor.batch_decode(ids, skip_special_tokens=True)
        del pv, ids, enc
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        return texts
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            LOGGER.warning('OOM — fallback to single')
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
            results = []
            for crop in crops:
                try: results.extend(trocr_batch_predict([crop]))
                except Exception: results.append('')
            return results
        return [''] * len(crops)
    except Exception as e:
        LOGGER.warning(f'trocr_batch: {e}')
        return [''] * len(crops)

print('✅ OCR Engine جاهز')

In [ ]:
# ============================================================
# الخلية 10: قاموس التصحيح
# ============================================================
def build_correction_dict() -> dict:
    if not CFG.feedback_csv.exists(): return {}
    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty: return {}
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o = normalize_text(row.get('original_text'))
            c = normalize_text(row.get('corrected_text'))
            if o and c and o != c: buckets[o][c] += 1
        result = {o: cnt.most_common(1)[0][0]
                  for o, cnt in buckets.items()
                  if cnt.most_common(1)[0][1] >= CFG.correction_min_votes}
        CFG.correction_dict_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), 'utf-8')
        return result
    except Exception as e:
        LOGGER.error(f'correction_dict: {e}')
        return {}

def load_correction_dict() -> dict:
    if CFG.correction_dict_path.exists():
        return json.loads(CFG.correction_dict_path.read_text('utf-8'))
    return build_correction_dict()

def apply_corrections(text: str, d: dict) -> str:
    return ' '.join(d.get(tok, tok) for tok in text.split()) if text else ''

def append_feedback(image_id, original, corrected, status='verified'):
    pd.DataFrame([{
        'timestamp': datetime.now().isoformat(),
        'image_id': image_id,
        'original_text': original,
        'corrected_text': corrected,
        'status': status,
    }]).to_csv(CFG.feedback_csv, mode='a', header=False, index=False, encoding='utf-8-sig')

correction_dict = build_correction_dict()
print(f'✅ قاموس التصحيح: {len(correction_dict)} إدخال')

In [ ]:
# ============================================================
# الخلية 11: Checkpoint
# ============================================================
def save_checkpoint(data: dict):
    CFG.checkpoint_file.write_text(json.dumps(data, ensure_ascii=False, indent=2), 'utf-8')

def load_checkpoint():
    return json.loads(CFG.checkpoint_file.read_text('utf-8')) if CFG.checkpoint_file.exists() else None

def clear_checkpoint():
    if CFG.checkpoint_file.exists(): CFG.checkpoint_file.unlink()

print('✅ Checkpoint جاهز')

In [ ]:
# ============================================================
# الخلية 12: معالجة PDF (نسخة v2 – صفحة بصفحة)
# ============================================================
def load_single_page(input_path: str, page_num: int):
    ext = Path(input_path).suffix.lower()
    if ext == '.pdf':
        imgs = convert_from_path(input_path, dpi=CFG.dpi, first_page=page_num, last_page=page_num)
        if not imgs: return None
        page_img = cv2.cvtColor(np.array(imgs[0]), cv2.COLOR_RGB2BGR)
        del imgs; gc.collect()
        return page_img
    elif ext in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}:
        return cv2.imread(input_path)
    return None

def get_page_count(input_path: str) -> int:
    ext = Path(input_path).suffix.lower()
    if ext == '.pdf':
        try:
            info = pdfinfo_from_path(input_path)
            return info.get('Pages', 1)
        except Exception:
            return 999
    return 1

def process_document_v2(input_path=None, start_page=None, end_page=None,
                         resume=True, adaptive=False, progress_cb=None) -> dict:
    global correction_dict
    input_path = str(input_path or CFG.pdf_path)
    run_id     = datetime.now().strftime('run_%Y%m%d_%H%M%S')
    t0         = time.time()
    correction_dict = load_correction_dict()

    ckpt = load_checkpoint() if resume else None
    sp = int(start_page or CFG.pages_start)
    ep = int(end_page   or CFG.pages_end)
    if ckpt and ckpt.get('input_path') == input_path:
        sp = int(ckpt.get('next_page', sp))
        LOGGER.info(f'استئناف من الصفحة {sp}')

    DB.insert_run(run_id, input_path)
    log_event('process_started_v2', {'run_id': run_id, 'input': input_path})
    DB.delete_pages(sp, ep)

    total_words, conf_acc = [], []
    page_stats = []
    with FileLock(CFG.lock_file, timeout=CFG.sync_lock_timeout):
        for actual_pg in range(sp, ep + 1):
            if progress_cb:
                progress_cb(actual_pg - sp + 1, ep - sp + 1, f'صفحة {actual_pg}/{ep}...')
            try:
                page_img = load_single_page(input_path, actual_pg)
            except Exception:
                break
            if page_img is None:
                break
            try:
                detections = easy_reader.readtext(page_img, detail=1)
            except Exception:
                detections = []
            binary, _ = preprocess_image(page_img, adaptive=adaptive)
            boxes = smart_segmentation(page_img, binary, detections)
            del binary
            boxes_info = match_boxes_detections(boxes, detections)
            need_trocr_idx, need_trocr_crops = [], []
            easy_results = []
            for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0:
                    easy_results.append(None); continue
                if easy_item is not None:
                    _, txt, conf = easy_item
                    easy_results.append(('easyocr', normalize_text(txt), float(conf)))
                    if float(conf) < CFG.easy_conf_threshold:
                        need_trocr_idx.append(i); need_trocr_crops.append(crop)
                else:
                    easy_results.append(None)
                    need_trocr_idx.append(i); need_trocr_crops.append(crop)
            trocr_texts = {}
            for b_start in range(0, len(need_trocr_crops), CFG.trocr_batch_size):
                batch_crops = need_trocr_crops[b_start:b_start+CFG.trocr_batch_size]
                texts = trocr_batch_predict(batch_crops)
                for k, txt in enumerate(texts):
                    trocr_texts[need_trocr_idx[b_start+k]] = txt
                del batch_crops
            del need_trocr_crops
            page_words = 0
            for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0: continue
                easy_res = easy_results[i]
                if easy_res and easy_res[2] >= CFG.easy_conf_threshold:
                    raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                elif i in trocr_texts and trocr_texts[i]:
                    raw, conf, src = trocr_texts[i], CFG.trocr_default_conf, 'trocr'
                    if easy_res and easy_res[2] > conf:
                        raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                elif easy_res:
                    raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
                else:
                    raw, conf, src = '', 0.0, 'none'
                corrected = apply_corrections(spell_correct(raw), correction_dict)
                _, buf = cv2.imencode('.png', crop)
                DB.insert_word(image_data=buf.tobytes(), predicted_text=corrected,
                               raw_text=raw, status='unverified', confidence=conf,
                               model_source=src, x=x, y=y, w=w, h=h,
                               page_num=actual_pg, run_id=run_id)
                total_words.append(1); conf_acc.append(conf); page_words += 1
                del buf
            page_stats.append({'page': actual_pg, 'words': page_words,
                               'avg_conf': round(float(np.mean(conf_acc[-page_words:])) if page_words else 0, 3)})
            del page_img, boxes_info, trocr_texts, easy_results
            gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
            save_checkpoint({'run_id': run_id, 'input_path': input_path,
                              'next_page': actual_pg + 1, 'words': sum(total_words)})
            log_event('page_done', {'page': actual_pg, 'words': page_words})

    duration = time.time() - t0
    avg_conf  = float(np.mean(conf_acc)) if conf_acc else 0.0
    pages_done = len(page_stats)
    clear_checkpoint()
    DB.finish_run(run_id, pages_done, sum(total_words), avg_conf)

    stats = {
        'run_id': run_id, 'ts': datetime.now().isoformat(),
        'input': input_path, 'pages': pages_done,
        'words': sum(total_words), 'avg_confidence': round(avg_conf, 4),
        'duration_sec': round(duration, 2),
        'page_stats': page_stats,
    }
    CFG.stats_json.write_text(json.dumps(stats, ensure_ascii=False, indent=2), 'utf-8')

    # runs history
    runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
    runs = pd.concat([runs, pd.DataFrame([{
        'run_id': run_id, 'timestamp': stats['ts'],
        'pages': pages_done, 'words': sum(total_words),
        'avg_conf': avg_conf, 'duration_sec': duration, 'status': 'completed',
    }])], ignore_index=True)
    runs.to_csv(CFG.runs_csv, index=False, encoding='utf-8-sig')

    if CFG.auto_export:
        stats['export'] = auto_export(run_id)

    log_event('process_finished_v2', stats)
    print(f'✅ اكتملت | {sum(total_words)} كلمة | {pages_done} صفحة | {duration:.1f}s')
    return stats

print('✅ معالج PDF (v2) جاهز')

In [ ]:
# ============================================================
# الخلية 13: التصدير التلقائي والنسخ الاحتياطي
# ============================================================
def auto_export(run_id: str, output_dir: Path = None) -> dict:
    output_dir = output_dir or (CFG.exports_dir / 'auto' / run_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(CFG.db_path) as c:
        df_all = pd.read_sql_query('SELECT * FROM handwriting_data ORDER BY page_num,y,x', c)
    df_verified = df_all[df_all['status'].isin(['verified','sentence_corrected'])]
    df_csv = df_all.drop(columns=['image_data'], errors='ignore')
    exported = {}

    p = output_dir / 'all_words.csv'
    df_csv.to_csv(p, index=False, encoding='utf-8-sig')
    exported['csv'] = str(p)

    px = output_dir / 'all_words.xlsx'
    with pd.ExcelWriter(px, engine='openpyxl') as w:
        df_csv.to_excel(w, sheet_name='All', index=False)
        for pg in sorted(df_csv['page_num'].dropna().unique()):
            df_csv[df_csv['page_num']==pg].to_excel(w, sheet_name=f'P{int(pg)}', index=False)
    exported['xlsx'] = str(px)

    lines_out = []
    for pg in sorted(df_all['page_num'].dropna().unique()):
        pw = df_all[df_all['page_num']==pg].sort_values(['y','x'])
        if pw.empty: continue
        curr = [pw.iloc[0].to_dict()]
        lgroups = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= 25: curr.append(row)
            else: lgroups.append(curr); curr = [row]
        lgroups.append(curr)
        for lg in lgroups:
            lang = detect_lang(' '.join(str(w.get('predicted_text','')) for w in lg))
            sl = sorted(lg, key=lambda k: k['x'], reverse=(lang=='ar'))
            lines_out.append(' '.join(str(w.get('predicted_text','')) for w in sl).strip())
    pt = output_dir / 'reconstructed_text.txt'
    pt.write_text('\n'.join(lines_out), 'utf-8')
    exported['text'] = str(pt)

    if not df_verified.empty:
        img_dir = output_dir / 'training_images'
        img_dir.mkdir(exist_ok=True)
        records = []
        for _, row in df_verified.iterrows():
            fname = f"img_{row['image_id']}.png"
            with open(img_dir / fname, 'wb') as f:
                f.write(row['image_data'])
            txt = normalize_text(row['predicted_text'])
            if txt: records.append({'image': fname, 'text': txt})
        pj = output_dir / 'training_data.jsonl'
        with open(pj, 'w', encoding='utf-8') as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        exported['jsonl']    = str(pj)
        exported['samples']  = len(records)

    summary = {'run_id': run_id, 'exported_at': datetime.now().isoformat(),
               'total_words': len(df_all), 'verified': len(df_verified),
               'dir': str(output_dir), 'files': exported}
    (output_dir / 'export_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    del df_all, df_verified, df_csv
    gc.collect()
    return summary

def create_backup() -> str:
    label  = datetime.now().strftime('%Y%m%d_%H%M%S')
    bk_dir = CFG.backups_dir / f'backup_{label}'
    bk_dir.mkdir(parents=True, exist_ok=True)
    for p in [CFG.db_path, CFG.feedback_csv, CFG.stats_json, CFG.correction_dict_path,
              _LOG_FILE, _EVENTS]:
        if Path(p).exists(): shutil.copy2(p, bk_dir / Path(p).name)
    return str(bk_dir)

print('✅ التصدير والنسخ الاحتياطي جاهزان')

In [ ]:
# ============================================================
# الخلية 14: إعادة بناء الجمل
# ============================================================
def reconstruct_sentences(verified_only=False, y_tolerance=25) -> pd.DataFrame:
    with sqlite3.connect(CFG.db_path) as c:
        q = "SELECT * FROM handwriting_data"
        if verified_only:
            q += " WHERE status IN ('verified','sentence_corrected')"
        q += " ORDER BY page_num,y,x"
        words = pd.read_sql_query(q, c)
    if words.empty: return pd.DataFrame()
    out = []
    for pg in words['page_num'].unique():
        pw = words[words['page_num']==pg].sort_values(['y','x'])
        curr = [pw.iloc[0].to_dict()]
        lines = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= y_tolerance: curr.append(row)
            else: lines.append(curr); curr = [row]
        lines.append(curr)
        for line in lines:
            preview = ' '.join(str(w.get('predicted_text','')) for w in line)
            lang    = detect_lang(preview)
            sl = sorted(line, key=lambda k: k['x'], reverse=(lang=='ar'))
            sentence = ' '.join(str(w.get('predicted_text','')) for w in sl).strip()
            out.append({'page': line[0]['page_num'], 'y_anchor': line[0]['y'],
                        'lang': lang, 'text': sentence,
                        'word_ids': [w['image_id'] for w in sl]})
    return pd.DataFrame(out)

print('✅ إعادة بناء الجمل جاهزة')

In [ ]:
# ============================================================
# الخلية 15: Metrics (WER/CER)
# ============================================================
def compute_metrics() -> dict:
    try:
        from jiwer import wer, cer
    except ImportError:
        return {'error': 'pip install jiwer'}
    words = DB.get_verified()
    valid = [w for w in words if w.get('raw_text','').strip() and w.get('predicted_text','').strip()]
    if len(valid) < 5:
        return {'wer': None, 'cer': None, 'samples': len(valid)}
    refs = [w['raw_text'].strip() for w in valid]
    hyps = [w['predicted_text'].strip() for w in valid]
    m = {'wer': round(wer(refs, hyps), 4), 'cer': round(cer(refs, hyps), 4),
         'samples': len(valid), 'timestamp': datetime.now().isoformat()}
    mdf = pd.DataFrame([m])
    if CFG.metrics_log.exists():
        mdf.to_csv(CFG.metrics_log, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        mdf.to_csv(CFG.metrics_log, index=False, encoding='utf-8-sig')
    return m

def plot_metrics_fig():
    if not CFG.metrics_log.exists(): return None
    df = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
    if df.empty: return None
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(df['wer'].dropna().values, label='WER', marker='o', color='#E53935')
    ax.plot(df['cer'].dropna().values, label='CER', marker='s', color='#1E88E5')
    ax.set_title('WER/CER'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

print('✅ Metrics جاهزة')

In [ ]:
# ============================================================
# الخلية 16: Active Learning
# ============================================================
def count_corrections() -> int:
    if not CFG.feedback_csv.exists(): return 0
    return len(pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig'))

def should_train(min_n=None) -> bool:
    return count_corrections() >= (min_n or CFG.al_min_corrections)

def reprocess_low_confidence(limit=None) -> int:
    limit = limit or CFG.al_reprocess_limit
    rows  = DB.get_low_confidence(CFG.low_conf_threshold, limit)
    if not rows: return 0
    done = 0
    for row in tqdm(rows, desc='إعادة معالجة'):
        try:
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            texts  = trocr_batch_predict([img_np])
            new    = texts[0].strip() if texts else ''
            if new:
                DB.update_word(row['image_id'], predicted_text=new); done += 1
            del img, img_np
        except Exception as e: LOGGER.debug(f'reprocess {row["image_id"]}: {e}')
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return done

print('✅ Active Learning جاهز')

In [ ]:
# ============================================================
# الخلية 17: Fine‑tuning (LoRA)
# ============================================================
def finetune_trocr_lora(min_samples=None, epochs=None, batch_size=None,
                         lr=None, progress_cb=None) -> dict:
    global trocr_model
    try:
        from peft import get_peft_model, LoraConfig, TaskType
        from torch.optim import AdamW
        from torch.utils.data import Dataset as TDS, DataLoader
        from torch.utils.tensorboard import SummaryWriter
    except ImportError as e:
        return {'error': str(e)}
    min_samples = min_samples or CFG.finetune_min_samples
    epochs      = epochs      or CFG.finetune_epochs
    batch_size  = batch_size  or CFG.finetune_batch_size
    lr          = lr          or CFG.finetune_lr
    verified = DB.get_verified()
    if len(verified) < min_samples:
        return {'error': f'عينات غير كافية: {len(verified)}/{min_samples}'}
    print(f'🚀 تدريب على {len(verified)} عينة')
    writer = SummaryWriter(log_dir=str(CFG.tb_dir / datetime.now().strftime('ft_%Y%m%d_%H%M%S')))
    try:
        import albumentations as A
        _aug = A.Compose([A.Rotate(limit=3, p=0.5), A.RandomBrightnessContrast(p=0.4), A.GaussNoise(p=0.3)])
        USE_AUG = True
    except ImportError: USE_AUG = False
    class HWDataset(TDS):
        def __init__(self, records): self.records = records
        def __len__(self): return len(self.records)
        def __getitem__(self, idx):
            row = self.records[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = np.array(img)
            if USE_AUG: img_np = _aug(image=img_np)['image']
            pv = trocr_processor(images=Image.fromarray(img_np), return_tensors='pt').pixel_values.squeeze(0)
            if CFG.use_fp16 and DEVICE.type == 'cuda': pv = pv.half()
            text = normalize_text(row['predicted_text'])
            lb = trocr_processor.tokenizer(text, return_tensors='pt', padding='max_length',
                                           max_length=CFG.max_text_length, truncation=True).input_ids.squeeze(0)
            lb[lb == trocr_processor.tokenizer.pad_token_id] = -100
            return {'pixel_values': pv, 'labels': lb}
    lora_cfg = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=CFG.lora_r, lora_alpha=CFG.lora_alpha,
                          target_modules=CFG.lora_targets, lora_dropout=CFG.lora_dropout)
    model = get_peft_model(trocr_model, lora_cfg)
    model.train()
    loader    = DataLoader(HWDataset(verified), batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)
    global_step, log = 0, []
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in loader:
            pv = batch['pixel_values'].to(DEVICE); lb = batch['labels'].to(DEVICE)
            out = model(pixel_values=pv, labels=lb)
            out.loss.backward(); optimizer.step(); optimizer.zero_grad()
            loss_val = float(out.loss.item())
            total_loss += loss_val
            writer.add_scalar('Loss/step', loss_val, global_step)
            global_step += 1
            del pv, lb, out
        avg_loss = total_loss / max(1, len(loader))
        msg = f'Epoch {epoch+1}/{epochs} | Loss={avg_loss:.4f}'
        log.append(msg); print(msg)
        if progress_cb: progress_cb(epoch+1, epochs, msg)
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    writer.close()
    model.save_pretrained(str(CFG.lora_path))
    trocr_processor.save_pretrained(str(CFG.lora_path))
    trocr_model = model
    log_event('finetune_done', {'epochs': epochs, 'samples': len(verified)})
    return {'status': 'done', 'log': log, 'lora_path': str(CFG.lora_path)}

def active_learning_cycle() -> str:
    global trocr_model
    if not should_train():
        return f'⏳ التصحيحات: {count_corrections()}/{CFG.al_min_corrections}'
    result = finetune_trocr_lora()
    if 'error' in result: return f'❌ {result["error"]}'
    if CFG.lora_path.exists():
        try:
            from peft import PeftModel
            trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_path)).to(DEVICE)
        except Exception as e:
            LOGGER.warning(f'reload lora: {e}')
    reprocessed = reprocess_low_confidence()
    m = compute_metrics()
    return f'✅ دورة AL اكتملت\nإعادة معالجة: {reprocessed} كلمة\nWER: {m.get("wer")} | CER: {m.get("cer")}'

def upload_to_hf(repo_id=None, hf_token=None) -> str:
    from huggingface_hub import HfApi, login
    repo_id  = repo_id  or CFG.hf_dataset_repo
    hf_token = hf_token or CFG.hf_token or _HF_TOKEN
    if not repo_id or not hf_token: return '❌ يحتاج repo_id و hf_token'
    stats = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    run_id = stats.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d")}')
    export_dir = CFG.exports_dir / 'auto' / run_id
    if not export_dir.exists(): auto_export(run_id)
    try:
        login(token=hf_token)
        api = HfApi()
        api.create_repo(repo_id=repo_id, repo_type='dataset', exist_ok=True)
        api.upload_folder(folder_path=str(export_dir), repo_id=repo_id, repo_type='dataset')
        return f'✅ https://huggingface.co/datasets/{repo_id}'
    except Exception as e: return f'❌ {e}'

print('✅ Fine‑tuning و AL جاهزان')

In [ ]:
# ============================================================
# الخلية 18: دليل الدراسة (Study Guide) — مع إصلاح HTML
# ============================================================
def _export_html_fixed(md_content: str, output_path: Path, title: str):
    html_lines = [f'''<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head><meta charset="UTF-8"><title>{title}</title>
<style>
body{{font-family:'Amiri','Segoe UI',sans-serif;max-width:900px;margin:0 auto;
padding:30px;line-height:1.8;color:#1a1a2e}}
h1{{text-align:center;color:#16213e;border-bottom:3px solid #0f3460;padding-bottom:15px}}
h2{{color:#0f3460;border-right:4px solid #e94560;padding-right:15px}}
h3{{color:#533483}}
table{{border-collapse:collapse;width:100%;margin:15px 0}}
th,td{{border:1px solid #ddd;padding:10px;text-align:right}}
th{{background:#0f3460;color:white}}
tr:nth-child(even){{background:#f8f9fa}}
ul{{margin:8px 0;padding-right:20px}}
li{{margin:4px 0}}
pre{{background:#f4f4f4;padding:12px;border-radius:6px;overflow-x:auto;font-family:monospace;direction:ltr;text-align:left}}
code{{background:#eee;padding:2px 5px;border-radius:3px;font-size:.9em}}
@media print{{table{{page-break-inside:avoid}};pre{{white-space:pre-wrap}}}}
</style></head><body>''']
    in_table = in_list = in_code = False
    code_buf = []
    for line in md_content.splitlines():
        s = line.strip()
        if s.startswith('```'):
            if in_code:
                html_lines.append('<pre><code>' + '\n'.join(code_buf) + '</code></pre>')
                code_buf = []; in_code = False
            else:
                if in_list: html_lines.append('</ul>'); in_list = False
                if in_table: html_lines.append('</table>'); in_table = False
                in_code = True
            continue
        if in_code:
            code_buf.append(s); continue
        if not s:
            if in_list: html_lines.append('</ul>'); in_list = False
            if in_table: html_lines.append('</table>'); in_table = False; continue
        if s.startswith('# '): html_lines.append(f'<h1>{s[2:]}</h1>')
        elif s.startswith('## '): html_lines.append(f'<h2>{s[3:]}</h2>')
        elif s.startswith('### '): html_lines.append(f'<h3>{s[4:]}</h3>')
        elif s.startswith('---'): html_lines.append('<hr>')
        elif s.startswith('|'):
            cells = [c.strip() for c in s.split('|') if c.strip()]
            if all(set(c) <= {'-',':'} for c in cells): continue
            if not in_table:
                if in_list: html_lines.append('</ul>'); in_list = False
                html_lines.append('<table>'); in_table = True
            html_lines.append('<tr>' + ''.join(f'<td>{c}</td>' for c in cells) + '</tr>')
        elif s.startswith('- ') or s.startswith('* '):
            if not in_list:
                if in_table: html_lines.append('</table>'); in_table = False
                html_lines.append('<ul>'); in_list = True
            html_lines.append(f'<li>{s[2:]}</li>')
        else:
            if in_list: html_lines.append('</ul>'); in_list = False
            if in_table: html_lines.append('</table>'); in_table = False
            html_lines.append(f'<p>{s}</p>')
    if in_list: html_lines.append('</ul>')
    if in_table: html_lines.append('</table>')
    if in_code and code_buf: html_lines.append('<pre><code>' + '\n'.join(code_buf) + '</code></pre>')
    html_lines.append('</body></html>')
    output_path.write_text('\n'.join(html_lines), 'utf-8')

def generate_study_guide(output_dir=None, include_mermaid=True,
                          include_flashcards=True, title=None) -> str:
    output_dir = Path(output_dir or CFG.study_guide_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    title = title or 'مرجع دراسة — مستخرج من الملاحظات اليدوية'
    words = DB.get_all()
    if not words:
        return '⚠️ لا توجد بيانات. شغّل المعالجة أولاً.'
    df = pd.DataFrame(words)
    pages = sorted(df['page_num'].dropna().unique())
    guide = [f'# {title}', f'تاريخ: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
             f'صفحات: {len(pages)}', '---']
    vocab_pairs = []
    for pg in pages:
        pw = df[df['page_num']==pg].sort_values(['y','x'])
        if pw.empty: continue
        guide.append(f'\n## صفحة {int(pg)}\n')
        curr = [pw.iloc[0].to_dict()]
        lgroups = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= 25: curr.append(row)
            else: lgroups.append(curr); curr = [row]
        lgroups.append(curr)
        table_rows = []
        for lg in lgroups:
            en_w = [str(w.get('predicted_text','')).strip() for w in lg
                    if all(ord(c)<128 for c in str(w.get('predicted_text','')).replace(' ',''))]
            ar_w = [str(w.get('predicted_text','')).strip() for w in lg
                    if any('\u0600'<=c<='\u06FF' for c in str(w.get('predicted_text','')))]
            if en_w or ar_w:
                table_rows.append({'EN': ' | '.join(en_w), 'AR': ' | '.join(ar_w)})
                if en_w and ar_w:
                    vocab_pairs.append({'en': ' '.join(en_w), 'ar': ' '.join(ar_w), 'page': int(pg)})
        if table_rows:
            guide.append('### جداول المصطلحات\n')
            guide.append('| إنجليزي | عربي |\n|---------|------|')
            for r in table_rows:
                guide.append(f"| {r['EN']} | {r['AR']} |")
            guide.append('')
        sent_lines = []
        for lg in lgroups:
            texts = [str(w.get('predicted_text','')).strip() for w in lg if w.get('predicted_text')]
            if not texts: continue
            txt = ' '.join(texts)
            lang = detect_lang(txt)
            sl = sorted(lg, key=lambda k: k['x'], reverse=(lang=='ar'))
            sent = ' '.join(str(w.get('predicted_text','')) for w in sl).strip()
            if sent: sent_lines.append(f'- {sent}')
        if sent_lines:
            guide.append('### الملاحظات\n')
            guide.extend(sent_lines); guide.append('')
    if include_mermaid and vocab_pairs:
        guide.append('\n---\n## خريطة المفردات\n')
        lines = ['mindmap', '  root((المصطلحات))']
        for v in vocab_pairs[:40]:
            en_id = v['en'].replace(' ','_')[:20]
            ar_id = v['ar'].replace(' ','_')[:20]
            lines.extend([f'    {en_id}[{v["en"]}]', f'      {ar_id}[{v["ar"]}]'])
        mermaid_code = '\n'.join(lines)
        guide.append(f'```mermaid\n{mermaid_code}\n```\n')
    if include_flashcards and vocab_pairs:
        guide.append('\n---\n## البطاقات التعليمية\n')
        import csv
        cards_path = output_dir / 'flashcards_anki.csv'
        with open(cards_path, 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.writer(f, delimiter=';')
            w.writerow(['front','back','tags'])
            for v in vocab_pairs:
                w.writerow([v['en'], v['ar'], f"page_{v['page']} EN-AR"])
                w.writerow([v['ar'], v['en'], f"page_{v['page']} AR-EN"])
        guide.append(f'تم توليد {len(vocab_pairs)*2} بطاقة → `{cards_path.name}`\n')
    content = '\n'.join(guide)
    md_path = output_dir / 'study_guide.md'
    md_path.write_text(content, 'utf-8')
    html_path = output_dir / 'study_guide.html'
    _export_html_fixed(content, html_path, title)
    print(f'✅ المرجع الدراسي في: {output_dir}')
    return content

print('✅ Study Guide جاهز')

In [ ]:
# ============================================================
# الخلية 19: Gradio helpers وحالة المراجعة
# ============================================================
_review_state = {'df': pd.DataFrame(), 'idx': 0}
_sent_state   = {'df': pd.DataFrame(), 'idx': 0}

def _word_row():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return None, '', '', 'لا توجد كلمات للمراجعة', 0.0, '0/0'
    row  = df.iloc[idx]
    pil  = Image.open(io.BytesIO(bytes(row['image_data'])))
    conf = float(row['confidence'] or 0)
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page_num']} "
            f"| {row['model_source']} | id={row['image_id']}")
    return (pil, str(row['predicted_text'] or ''),
            str(row['raw_text'] or ''), info, conf, f'{idx+1}/{len(df)}')

def load_word_review():
    with sqlite3.connect(CFG.db_path) as c:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY confidence ASC, image_id ASC", c)
    _review_state['df'] = df; _review_state['idx'] = 0
    return _word_row()

def word_confirm(corrected_text: str):
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty: return _word_row()
    row  = df.iloc[idx]; rid = int(row['image_id'])
    orig = normalize_text(row['predicted_text'])
    corr = normalize_text(corrected_text)
    DB.update_word(rid, predicted_text=corr, status='verified')
    DB.log_review(rid, orig, corr, 'verified')
    if orig != corr: append_feedback(rid, orig, corr, 'verified')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()

def word_delete():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty: return _word_row()
    rid = int(df.iloc[idx]['image_id'])
    DB.delete_word(rid); DB.log_review(rid, '', '', 'delete')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()

def word_prev():
    if not _review_state['df'].empty:
        _review_state['idx'] = max(0, _review_state['idx']-1)
    return _word_row()

def word_next():
    df = _review_state['df']
    if not df.empty:
        _review_state['idx'] = min(len(df)-1, _review_state['idx']+1)
    return _word_row()

def word_undo_fixed():
    last_event = DB.get_last_review_event()
    if not last_event or last_event.get('image_id') is None:
        return _word_row()
    event_id      = last_event['id']
    image_id      = last_event['image_id']
    original_text = last_event['original_text']
    corrected_text= last_event['corrected_text']
    DB.update_word(image_id, predicted_text=original_text, status='unverified')
    if CFG.feedback_csv.exists():
        df_fb = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        mask  = ~((df_fb['image_id'].astype(str) == str(image_id)) &
                  (df_fb['original_text'].astype(str) == str(original_text)) &
                  (df_fb['corrected_text'].astype(str) == str(corrected_text)))
        df_fb[mask].to_csv(CFG.feedback_csv, index=False, encoding='utf-8-sig')
    DB.delete_review_event(event_id)
    global correction_dict
    correction_dict = build_correction_dict()
    # إعادة إدراج الكلمة المسترجعة في الواجهة
    with sqlite3.connect(CFG.db_path) as c:
        c.row_factory = sqlite3.Row
        row = c.execute("SELECT * FROM handwriting_data WHERE image_id=?", (image_id,)).fetchone()
    if row:
        row_dict = dict(row)
        df  = _review_state['df']
        idx = _review_state['idx']
        new_row_df = pd.DataFrame([row_dict])
        df_new = pd.concat([df.iloc[:idx], new_row_df, df.iloc[idx:]]).reset_index(drop=True)
        _review_state['df'] = df_new
    return _word_row()

def word_copy_raw(raw_text: str):
    return raw_text

def _sent_row():
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty: return '', 'لا توجد جمل', '0/0'
    row = df.iloc[idx]
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page']} "
            f"| {row['lang']} | {len(row['word_ids'])} كلمة")
    return row['text'], info, f'{idx+1}/{len(df)}'

def load_sent_review():
    _sent_state['df'] = reconstruct_sentences(verified_only=False)
    _sent_state['idx'] = 0
    return _sent_row()

def sent_save(corrected: str):
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty: return _sent_row()
    row  = df.iloc[idx]
    orig = normalize_text(row['text'])
    corr = normalize_text(corrected)
    ts   = datetime.now().isoformat()
    with sqlite3.connect(CFG.db_path) as c:
        for wid in row['word_ids']:
            c.execute("UPDATE handwriting_data SET status='sentence_corrected', updated_at=? WHERE image_id=?",
                      (ts, int(wid)))
        c.commit()
    append_feedback(None, orig, corr, f"sent_p{row['page']}")
    _sent_state['idx'] = min(idx+1, max(0, len(df)-1))
    return _sent_row()

def sent_prev():
    if not _sent_state['df'].empty:
        _sent_state['idx'] = max(0, _sent_state['idx']-1)
    return _sent_row()

def sent_next():
    df = _sent_state['df']
    if not df.empty:
        _sent_state['idx'] = min(len(df)-1, _sent_state['idx']+1)
    return _sent_row()

def build_dashboard_fig():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    counts = DB.count_by_status()
    if counts:
        axes[0].bar(list(counts.keys()), list(counts.values()),
                    color=['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0'])
        axes[0].set_title('توزيع الحالات')
    else: axes[0].text(0.5,0.5,'لا بيانات', ha='center')
    if CFG.runs_csv.exists():
        runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
        if not runs.empty:
            vals = pd.to_numeric(runs['words'], errors='coerce').fillna(0)
            axes[1].plot(vals.values, marker='o', color='#2196F3')
        else: axes[1].text(0.5,0.5,'لا سجلات', ha='center')
    else: axes[1].text(0.5,0.5,'لا سجلات', ha='center')
    if CFG.metrics_log.exists():
        mdf = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
        if not mdf.empty:
            axes[2].plot(mdf['wer'].dropna().values, label='WER', color='#E53935', marker='o')
            axes[2].plot(mdf['cer'].dropna().values, label='CER', color='#1E88E5', marker='s')
            axes[2].legend(); axes[2].set_ylim(0,1)
        else: axes[2].text(0.5,0.5,'لا مقاييس', ha='center')
    else: axes[2].text(0.5,0.5,'لا مقاييس بعد', ha='center')
    axes[2].set_title('WER / CER')
    plt.tight_layout()
    return fig

def get_dashboard_text() -> str:
    lines = ['## 📊 ملخص المشروع']
    counts = DB.count_by_status()
    lines.append(f"**إجمالي الكلمات:** {sum(counts.values())}")
    for k,v in counts.items(): lines.append(f'- {k}: **{v}**')
    if CFG.stats_json.exists():
        s = json.loads(CFG.stats_json.read_text())
        lines.append(f"\n**آخر تشغيل:** {s.get('run_id','—')} | صفحات: {s.get('pages')} | كلمات: {s.get('words')} | ثقة: {s.get('avg_confidence')}")
    return '\n'.join(lines)

# دوال ربط Gradio
def do_process(inp, sp, ep, resume, adaptive, progress=gr.Progress()):
    ep_val = int(ep) if ep and str(ep).strip() else CFG.pages_end
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    try:
        stats = process_document_v2(inp, int(sp), ep_val, resume, adaptive, cb)
        return f"✅ اكتملت\n```json\n{json.dumps(stats, ensure_ascii=False, indent=2)}\n```"
    except Exception as e: return f'❌ {e}'

def do_export():
    s = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    rid = s.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
    return f"✅\n```json\n{json.dumps(auto_export(rid), ensure_ascii=False, indent=2)}\n```"

def do_backup(): return f'✅ `{create_backup()}`'

def do_metrics():
    m = compute_metrics()
    if 'error' in m: return m['error'], None
    return (f"**WER:** {m['wer']} | **CER:** {m['cer']} | **عينات:** {m['samples']}", build_dashboard_fig())

def do_finetune(min_s, ep, bs, lr, progress=gr.Progress()):
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    result = finetune_trocr_lora(int(min_s), int(ep), int(bs), float(lr), cb)
    if 'error' in result: return f"❌ {result['error']}"
    return '✅ اكتمل\n' + '\n'.join(result.get('log',[]))

def do_al(): return active_learning_cycle()
def do_upload(r, t): return upload_to_hf(r or None, t or None)

print('✅ Gradio helpers جاهزة')

In [ ]:
# ============================================================
# الخلية 20: واجهة Gradio (7 تبويبات)
# ============================================================
def build_app():
    with gr.Blocks(title='Arabic Handwriting OCR — v5',
                   theme=gr.themes.Soft(primary_hue='indigo')) as app:
        gr.Markdown(
            '# 🖋️ Arabic Handwriting OCR — v5 النهائية\n'
            '> TrOCR + EasyOCR | LoRA | Active Learning | Study Guide | Multi‑Device')

        # تبويب 1: معالجة
        with gr.Tab('⚙️ المعالجة'):
            with gr.Row():
                inp_path   = gr.Textbox(label='مسار الملف', value=CFG.pdf_path)
                start_page = gr.Number(label='من الصفحة', value=CFG.pages_start, precision=0)
                end_page   = gr.Textbox(label='إلى الصفحة', value=str(CFG.pages_end))
            with gr.Row():
                resume_cb   = gr.Checkbox(label='استئناف', value=True)
                adaptive_cb = gr.Checkbox(label='Adaptive Threshold', value=False)
            run_btn = gr.Button('🚀 ابدأ', variant='primary')
            run_out = gr.Markdown()
            run_btn.click(do_process, [inp_path, start_page, end_page, resume_cb, adaptive_cb], run_out)

        # تبويب 2: مراجعة الكلمات
        with gr.Tab('🔍 مراجعة الكلمات'):
            load_w = gr.Button('📥 تحميل')
            word_info = gr.Markdown('—')
            word_prog = gr.Textbox(label='التقدم', interactive=False)
            word_img  = gr.Image(label='الصورة', type='pil', height=160)
            word_raw  = gr.Textbox(label='النص الخام', interactive=False)
            copy_btn  = gr.Button('📄 نسخ الخام', variant='secondary', size='sm')
            word_edit = gr.Textbox(label='النص المعدل ✏️')
            conf_sl   = gr.Slider(0,1, label='الثقة', interactive=False)
            with gr.Row():
                prev_w = gr.Button('⬅ السابق')
                undo_w = gr.Button('↩️ تراجع', variant='secondary')
                conf_w = gr.Button('✅ تأكيد', variant='primary')
                del_w  = gr.Button('🗑 حذف', variant='stop')
                next_w = gr.Button('التالي ➡')
            _wo = [word_img, word_edit, word_raw, word_info, conf_sl, word_prog]
            load_w.click(load_word_review, outputs=_wo)
            copy_btn.click(word_copy_raw, [word_raw], [word_edit])
            conf_w.click(word_confirm, [word_edit], _wo)
            del_w.click(word_delete, outputs=_wo)
            prev_w.click(word_prev, outputs=_wo)
            next_w.click(word_next, outputs=_wo)
            undo_w.click(word_undo_fixed, outputs=_wo)

        # تبويب 3: مراجعة الجمل
        with gr.Tab('📝 مراجعة الجمل'):
            load_s = gr.Button('📥 تحميل الجمل')
            sent_info = gr.Markdown('—')
            sent_prog = gr.Textbox(label='التقدم', interactive=False)
            sent_edit = gr.Textbox(label='الجملة ✏️', lines=3)
            with gr.Row():
                prev_s = gr.Button('⬅ السابق')
                save_s = gr.Button('✅ حفظ', variant='primary')
                next_s = gr.Button('التالي ➡')
            _so = [sent_edit, sent_info, sent_prog]
            load_s.click(load_sent_review, outputs=_so)
            save_s.click(sent_save, [sent_edit], _so)
            prev_s.click(sent_prev, outputs=_so)
            next_s.click(sent_next, outputs=_so)

        # تبويب 4: Dashboard
        with gr.Tab('📊 Dashboard'):
            refresh_btn = gr.Button('🔄 تحديث')
            dash_text   = gr.Markdown()
            dash_plot   = gr.Plot()
            with gr.Row():
                export_btn  = gr.Button('📤 تصدير', variant='secondary')
                backup_btn  = gr.Button('💾 نسخة', variant='secondary')
                metrics_btn = gr.Button('📐 WER/CER', variant='secondary')
            export_out  = gr.Markdown()
            backup_out  = gr.Markdown()
            metrics_out = gr.Markdown()
            metrics_plt = gr.Plot()
            refresh_btn.click(lambda: (get_dashboard_text(), build_dashboard_fig()),
                              outputs=[dash_text, dash_plot])
            export_btn.click(do_export, outputs=export_out)
            backup_btn.click(do_backup, outputs=backup_out)
            metrics_btn.click(do_metrics, outputs=[metrics_out, metrics_plt])
            app.load(lambda: (get_dashboard_text(), build_dashboard_fig()),
                     outputs=[dash_text, dash_plot])

        # تبويب 5: Fine‑tuning & AL
        with gr.Tab('🧠 Fine‑tuning & AL'):
            with gr.Row():
                ft_min = gr.Number(label='الحد الأدنى للعينات', value=CFG.finetune_min_samples, precision=0)
                ft_ep  = gr.Number(label='Epochs', value=CFG.finetune_epochs, precision=0)
                ft_bs  = gr.Number(label='Batch size', value=CFG.finetune_batch_size, precision=0)
                ft_lr  = gr.Number(label='Learning rate', value=CFG.finetune_lr)
            ft_btn = gr.Button('🔥 ابدأ Fine‑tuning', variant='primary')
            ft_out = gr.Markdown()
            ft_btn.click(do_finetune, [ft_min, ft_ep, ft_bs, ft_lr], ft_out)
            gr.Markdown('---\n### 🔁 Active Learning')
            al_btn = gr.Button('🧠 دورة AL', variant='secondary')
            al_out = gr.Markdown()
            al_btn.click(do_al, outputs=al_out)

        # تبويب 6: Study Guide
        with gr.Tab('📚 دليل الدراسة'):
            sg_btn = gr.Button('📖 توليد المرجع الدراسي', variant='primary')
            sg_out = gr.Markdown()
            sg_btn.click(lambda: generate_study_guide(), outputs=sg_out)

        # تبويب 7: إعدادات
        with gr.Tab('⚙️ الإعداد'):
            gr.Markdown('### ترحيل البيانات القديمة')
            migrate_btn = gr.Button('🔄 بدء الترحيل', variant='secondary')
            migrate_out = gr.Markdown()
            migrate_btn.click(lambda: json.dumps(SMART_MIGRATOR.migrate(delete_after=True), ensure_ascii=False, indent=2),
                              outputs=migrate_out)
            gr.Markdown('---\n### رفع إلى HuggingFace')
            hf_repo  = gr.Textbox(label='Repo ID', value=CFG.hf_dataset_repo)
            hf_token = gr.Textbox(label='HF Token', type='password')
            hf_btn   = gr.Button('🚀 رفع', variant='primary')
            hf_out   = gr.Markdown()
            hf_btn.click(do_upload, [hf_repo, hf_token], hf_out)

    return app

print('✅ واجهة Gradio جاهزة')

In [ ]:
# ============================================================
# الخلية 21: التشغيل
# ============================================================
def launch():
    app = build_app()
    app.launch(share=CFG.gradio_share, server_port=0, quiet=False)
    return app

print('✅✅✅ كل شيء جاهز — v5 Final ✅✅✅')
print()
print('launch() ← تشغيل UI')
print('process_document_v2() ← معالجة بدون UI')
print('SMART_MIGRATOR.migrate(delete_after=True) ← ترحيل القديم')
print('finetune_trocr_lora() ← تحسين النموذج')
print('generate_study_guide() ← دليل الدراسة')

if IN_COLAB:
    launch()
else:
    print('للتشغيل المحلي: launch()')